In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

# Load the data
df = pd.read_csv(
    '~/Documents/projects/loan-default-fairness/data/accepted_2007_to_2018Q4.csv',
    low_memory=False
)

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")

In [ ]:
# Load the approved-loans data dictionary
data_dict = pd.read_csv(
    '~/Documents/projects/loan-default-fairness/data/Lending Club Data Dictionary Approved.csv',  
    encoding='latin-1'
)
print(f"Dictionary has {len(data_dict)} entries")
data_dict.head(20)

In [ ]:
# Keep only the two useful columns, drop empty rows
data_dict = data_dict[['LoanStatNew', 'Description']].dropna(subset=['LoanStatNew'])
print(f"Clean dictionary: {len(data_dict)} field definitions\n")

# List every column in the actual dataset
print("Columns in our dataset:")
for i, col in enumerate(df.columns):
    print(f"{i:3d}: {col}")

In [ ]:
# Re-create the filtered dataset with target
# Filter to completed loans and build target
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

# Derive features from the date columns before we drop them
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['issue_year'] = df['issue_d'].dt.year

df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
df['credit_history_months'] = (
    (df['issue_d'] - df['earliest_cr_line']).dt.days / 30.44
).round().astype('Int64')

print(f"Rows: {len(df):,}, default rate: {df['default'].mean():.2%}")

In [ ]:
leakage_payment = [
    'pymnt_plan', 'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
    'total_rec_int', 'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
    'next_pymnt_d', 'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low',
]

leakage_hardship = [
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',
]

leakage_settlement = [
    'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term',
]

junk = ['id', 'member_id', 'url', 'policy_code']
text_features = ['emp_title', 'desc', 'title']
borderline = ['disbursement_method']

# Raw date columns we already derived features from, plus the target source
post_processing = ['loan_status', 'issue_d', 'earliest_cr_line']

cols_to_drop = (leakage_payment + leakage_hardship + leakage_settlement
                + junk + text_features + borderline + post_processing)

print(f"Columns before: {df.shape[1]}")
df_model = df.drop(columns=cols_to_drop)
print(f"Columns after:  {df_model.shape[1]}")
print(f"Dropped:        {df.shape[1] - df_model.shape[1]}")

In [ ]:
# Install pyarrow first if needed: in terminal, run `uv add pyarrow`
df_model.to_parquet('../data/loans_clean.parquet', index=False)
print(f"Saved clean dataset: {df_model.shape[0]:,} rows, {df_model.shape[1]} columns")